# Ingestion and chunking
Documents containing construction site visit reports are loaded, normalized and chunked. Then these chunks are saved to a JSON file.

In [ ]:
# Install packages for working with DOCX files (reading and text extraction)
%pip install python-docx
%pip install docx2txt

In [ ]:
# Import libraries for document processing and text handling
import json
import re
import uuid
from datetime import datetime
from pathlib import Path

import docx2txt
from docx import Document
from google.colab import drive
from zipfile import ZipFile, BadZipFile
from transformers import AutoTokenizer


In [ ]:
# Functions for chunking

MIN_CHUNK_LENGTH = 40
MIN_ALPHA_RATIO = 0.65

def normalize_text(text):
    """
    Normalize text into a canonical form suitable for processing.

    Currently:
    - Collapses consecutive whitespace into a single space.
    - Removes leading and trailing whitespace.
    """

    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def is_valid_chunk(chunk,
                 min_chunk_length=MIN_CHUNK_LENGTH,
                 min_alpha_ratio=MIN_ALPHA_RATIO):
    """
    Return whether the chunk meets the minimum quality requirements.

    A valid chunk must contain:
    - A minimum number of words.
    - A minimum ratio of alphabetic characters.
    """

    word_count = len(chunk.split())
    if word_count < min_chunk_length:
        return False

    alpha_ratio = sum(c.isalpha() for c in chunk) / max(len(chunk), 1)
    if alpha_ratio < min_alpha_ratio:
        return False

    return True

def is_valid_docx(path: Path) -> bool:
    """
    Check if file is a valid .docx (ZIP with word/document.xml).
    """
    try:
        with ZipFile(path) as z:
            return "word/document.xml" in z.namelist()
    except BadZipFile:
        return False
    except Exception:
        return False

def extract_docx_text(docx_path):
    """
    Extract the textual content from a Word document.

    This helper is intended for linear text extraction and is used by
    the character-based and token-based chunking strategies.
    """
    return docx2txt.process(docx_path)


def chunk_docx_char_based(docx_path,
                          chunk_size,
                          overlap,
                          avoid_break_words=False,
                          strict_stride=True):
    """
    Split text into overlapping chunks using a character-based sliding window.

    Chunks are optionally adjusted to avoid breaking words and are filtered
    using a validation function before being returned.
    """

    if overlap >= chunk_size:
        raise ValueError("Overlap must be smaller than chunk_size")

    chunks = []
    stride = max(1, chunk_size - overlap)
    start_chunk_pos = 0

    docx_text = normalize_text(extract_docx_text(docx_path))
    docx_text_length = len(docx_text)

    while start_chunk_pos < docx_text_length:

        # Tentative end of chunk
        end_chunk_pos = min(start_chunk_pos + chunk_size, docx_text_length)

        # Adjust end to last whitespace to avoid breaking words
        if avoid_break_words and end_chunk_pos < docx_text_length:
              space_pos = docx_text.rfind(" ", start_chunk_pos, end_chunk_pos)
              if space_pos != -1 and space_pos > start_chunk_pos:
                  end_chunk_pos = space_pos

        # Extract chunk within computed boundaries
        chunk_text = docx_text[start_chunk_pos:end_chunk_pos].strip()

        if is_valid_chunk(chunk_text):
            chunks.append(chunk_text)

        # Move forward by stride
        start_chunk_pos += stride

        # Optional alignment for not starting next chunk in the middle of a word
        if avoid_break_words and not strict_stride:
            while start_chunk_pos < docx_text_length and docx_text[start_chunk_pos] != " ":
                start_chunk_pos += 1

    return chunks


def chunk_docx_token_based(docx_path, tokenizer, chunk_size, overlap):
    """
    Split a document into overlapping chunks using a token-based sliding window.

    Chunks are created at the token level rather than character level, allowing
    more linguistically meaningful segmentation.
    """

    if overlap >= chunk_size:
        raise ValueError("Overlap must be smaller than chunk_size")

    chunks = []
    stride = chunk_size - overlap

    docx_text = normalize_text(extract_docx_text(docx_path))
    docx_token_ids = tokenizer.encode(docx_text, add_special_tokens=False)

    for i in range(0, len(docx_token_ids), stride):
        chunk_token_ids = docx_token_ids[i:i + chunk_size]

        chunk_text = tokenizer.decode(chunk_token_ids)

        if is_valid_chunk(chunk_text):
            chunks.append(chunk_text)

    return chunks


def chunk_docx_table_row_based(docx_path, separator=" || "):
    """
    Extract Word table rows as semantic chunks.

    Each row is treated as an independent chunk composed of multiple cells.
    Cells are normalized and joined using a separator, then validated using
    the standard chunk validation pipeline.
    """

    doc = Document(docx_path)
    chunks = []

    for table in doc.tables:
        for row in table.rows:

            cleaned_cells = []
            seen = set()

            for cell in row.cells:
                text = normalize_text(cell.text)

                # Skip empty cells after normalization
                if not text:
                    continue

                # Avoids duplicates of the same cell content
                if text in seen:
                    continue
                seen.add(text)

                cleaned_cells.append(text)

            # Skip empty rows
            if not cleaned_cells:
                continue

            chunk_text = separator.join(cleaned_cells)

            # Apply global validation pipeline
            if is_valid_chunk(chunk_text):
                chunks.append(chunk_text)

    return chunks

In [ ]:
# Function for chunking all documents contained in a directory

CHUNK_STRATEGIES = {
    "char": {
        "chunk_fn": chunk_docx_char_based,
        "req_params": ["chunk_size", "overlap"]
    },
    "token": {
        "chunk_fn": chunk_docx_token_based,
        "req_params": ["tokenizer", "chunk_size", "overlap"]
    },
    "table": {
        "chunk_fn": chunk_docx_table_row_based,
        "req_params": []
    }
}

def chunk_all_docx(docx_dir, strategy_key, **kwargs):
    """
    Extract chunks from all Word documents in a directory using a selected chunking strategy.

    Each document is processed independently, and its chunks are enriched with metadata
    (document identifier, chunk index, unique identifier, and embedding placeholder).
    The output is serialized into a JSON file for downstream embedding and retrieval pipelines.
    """

    # Resolve strategy
    if strategy_key not in CHUNK_STRATEGIES:
        raise ValueError(f"Unsupported strategy: {strategy_key}")
    strategy = CHUNK_STRATEGIES[strategy_key]
    chunk_fn = strategy["chunk_fn"]

    # Validates strategy parameters
    req_params = strategy["req_params"]
    for param in req_params:
        if param not in kwargs:
            raise ValueError(f"Missing required param: {param}")

    # Chunk generation metatada
    chunking_process = {
        "process_id": str(uuid.uuid4()),
        "timestamp": datetime.now().isoformat(),
        "strategy": strategy_key
    }
    if "chunk_size" in kwargs:
        chunking_process["chunk_size"] = kwargs["chunk_size"]
    if "overlap" in kwargs:
        chunking_process["overlap"] = kwargs["overlap"]
    if "avoid_break_words" in kwargs:
        chunking_process["avoid_break_words"] = kwargs["avoid_break_words"]
    if "strict_stride" in kwargs:
        chunking_process["strict_stride"] = kwargs["strict_stride"]
    if "tokenizer" in kwargs:
        chunking_process["tokenizer"] = kwargs["tokenizer"].name_or_path

    tst = str(int(datetime.now().timestamp()))
    json_filename = f"chunks_{strategy_key}_{tst}.json"
    json_file_path = f"{docx_dir}/{json_filename}"

    # Get documents
    docx_paths = list(Path(docx_dir).glob("*.docx")) + list(Path(docx_dir).glob("*.doc"))

    chunk_info_list = []
    chunk_index = 0
    for docx_path in docx_paths:

        # Use filename (without extension) as document identifier
        docx_id = docx_path.stem
        if not is_valid_docx(docx_path):
            print(f"{docx_id} is not a docx file")
            continue
        else:
            print(docx_id)

        #  Obtain a document's chunks
        docx_chunks = chunk_fn(docx_path, **kwargs)
        if not docx_chunks:
            continue

        # Collect document chunks and assign each an RFC 9562 UUID
        for docx_chunk in docx_chunks:
            chunk_info_list.append({
                "docx_id": docx_id,
                "chunk_id": str(uuid.uuid4()),
                "chunk_index": chunk_index,
                "chunk_text": docx_chunk,
                "chunk_length": len(docx_chunk)
                })
            chunk_index += 1

    # Output to be serialized
    output = {
        "chunking_process": chunking_process,
        "chunks": chunk_info_list
    }

    # Save document chunks to a JSON file
    with open(json_file_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=4, ensure_ascii=False)


In [ ]:
# Mount Google Drive for file access
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DOCX_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project"
docs_paths = list(Path(DOCX_DIR).glob("*.docx")) + list(Path(DOCX_DIR).glob("*.doc"))

# Character-based chunking
chunk_all_docx(docx_dir=DOCX_DIR, strategy_key="char", chunk_size=1000, overlap=200, avoid_break_words=True, strict_stride=True)
chunk_all_docx(docx_dir=DOCX_DIR, strategy_key="char", chunk_size=1000, overlap=200, avoid_break_words=True, strict_stride=False)

# Token-based chunking with Multilingual E5 Base
model_name = "intfloat/multilingual-e5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
chunk_all_docx(docx_dir=DOCX_DIR, strategy_key="token", tokenizer=tokenizer, chunk_size=500, overlap=100)

# Table-rows-based chunking
chunk_all_docx(docx_dir=DOCX_DIR, strategy_key="table")

In [ ]:
list(Path(DOCX_DIR).glob("chunks_*.json"))